<a href="https://colab.research.google.com/github/Bast1-py/Experience-Developing-Projects/blob/main/Text_Clustering_and_Semantic_Similarity_using_NLP_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.display import Image, HTML, display, display_html
source_tautan = "https://skills.network/logos/SN_web_lightmode.png"
HTML(f'<div style="text-align: center;"><img src="{source_tautan}" style="max-width: 600px; height: auto;"></div>')

# **1. Introduction:**

**Sentence-BERT** adalah modifikasi dari framework BERT untuk memungkinkan perbandingan **Large-scale semantic similarity, clustering dan information retrieval.**

**Sentence-BERT** atau **SBERT** menggunakan struktur jaringan **siamese** dan **triplet** untuk menghasilkan embedding kalimat yang bermakna secara semantik yang dapat dibandingkan menggunakan **cosine-similarity**. **siamese network** terdiri dari dua atau lebih sub-jaringan identik, dimana kedua sub-jaringan tersebut berbagi bobot yang sama. pembaruan bobot dicerminkan diseluruh sub-jaringan selama pelatihan. berikut dibawah ini diagram arsitektur SBERT dengan fungsi tujuan klasifikasi:



In [ ]:
from IPython.display import Image, HTML, display, display_html
source_tautan = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-GPXX068IEN/images/SBERT.png"
HTML(f'<div style="text-align: center;"><img src="{source_tautan}" style="max-width: 450px; height: auto;"></div>')

diagram diatas menunjukkan bahwa SBERT menggunakan 2 model BERT "twin" dan menambahkan operasi pooling pada output dari kedua model BERT tersebut. **The pooling layers** menciptakan embedding kalimat berukuran tetap untuk kalimat input.

embedding kalimat adalah bidang yang banyak dipelajari dengan puluhan metode yang diusulkan. metode embedding kalimat sebelumnya seperti **InferSent** dan **Universal Sentence Encoder** dilatih dari random initializations.

para penulis **Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks** menggunakan jaringan **BERT** dan **RoBERTa** yang telah dilatih sebelumnya dan melakukan fine-tuning untuk menghasilkan embedding kalimat, yang secara signifikan menbgurangi waktu pelatihan (SBERT dapat di-tuning dalam waktu kurang dari 20 menit) sambil menghasilkan hasil yang lebih baik daripada metode embedding kalimat yang sebanding.

SBERT memiliki library python bernama sentence_transformers yang dapat digunakan untuk menghitung embedding kalimat untuk lebih dari 100 bahasa. embedding ini kemudian dapat dibandingkan, misalnya, dengan cosine similarity untuk menemukan kalimat dengan makna yang serupa. ini dapat berguna untuk kesamaan teks semantik, semantic search, atau penambahan parafrasa.

pustaka ini menawarkan koleksi dari [pretrained models](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkGuidedProjectsIBMGPXX068IEN1371-2022-01-01) untuk berbagai tugas. misalnya **all-mpnet-base-v2:** memberikan kualitas terbaik, sementara **all-MiniLM-L6-v2:** 5 kali lebih cepat dan tetap menawarkan kualitas yang baik.

# **2. Importing Required Libraries**

In [ ]:
# pip install adjustText

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

%matplotlib inline

from adjustText import adjust_text
from numpy.linalg import norm
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from scipy.stats import gaussian_kde
from sentence_transformers import SentenceTransformer, util

import warnings
warnings.filterwarnings("ignore")

# **3. Defining Helper Functions**

In [ ]:
def plotter(
    x,
    y,
    title):
  plt.plot(x,y)
  plt.xlabel('X')
  plt.ylabel('Y')
  plt.title(title)
  plt.show()

# **4. SBERT For Sentence Embeddings.**

In [ ]:
sentences = [
    'This Framework Generates embeddings For Each Input Sentence.',
    'Sentences Are Passed as a List of String.',
    'The Quick Brown Fox Jumps Over The Lazy Dog.',
    'Sequential Models in Keras are Defined as a Sequence of Layers.',
    'You Need to Ensure The Input Layer Has The Right Number of Inputs.',
    'What Best For Evaluation Metrics is That You Decide The Optimum Number of Layers and The Parameters and Steps in Each Layer.',
    'A Heuristics Approach is Also Used.',
    'The Best Network Structure is Found Through a Process of Trial-and-Error experimentation.',
    'Generally, You Need a Network Large Enough to Capture The Structure of The Problem.',
    'Having Defined The Model in Terms of Layers, You Need to Declare The Loss Function, The Optimizer, and The Evaluation Metrics.',
    'When The Model is Proposed, The Initial Weight and Bias Values are Assumed to be 0 or 1, a Random Normally Distributed Number, or Any Other Convenient Numbers.',
    'The Journey From Initial Values to Optimal Values Needs a Motivation, Which Will Minimize The Cost Function/Lost Function.'
]

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [ ]:
embeddings = model.encode(
    sentences,
    convert_to_numpy=True
)

embeddings.shape

(12, 384)

In [ ]:
embeddings[0][:110]

array([-0.01195315, -0.05562935, -0.00824257,  0.00889046,  0.02768427,
        0.11398811,  0.0146988 , -0.0318959 ,  0.04145178, -0.08188555,
        0.01413271, -0.02033361,  0.04077511,  0.02262852, -0.04784385,
        0.07633466, -0.00329894,  0.04053929, -0.04151126, -0.09595738,
        0.00275886,  0.06117341,  0.05333673, -0.0446904 , -0.05076976,
        0.04299559, -0.05940743,  0.00790485,  0.10338719,  0.01843008,
        0.02685579, -0.02852802,  0.03100763,  0.07762937, -0.0013566 ,
        0.01041263, -0.01150964,  0.0383571 , -0.04920458, -0.01525103,
       -0.03559209, -0.00363358,  0.02812768,  0.03145439,  0.06721427,
       -0.03931508, -0.10906401, -0.01975164, -0.02292146,  0.04114951,
       -0.08468425, -0.05844709, -0.00620921,  0.02703554, -0.00866036,
        0.02391588, -0.01988747, -0.01934548,  0.02159742, -0.06250764,
       -0.05213042, -0.0500552 , -0.01291964,  0.0481705 ,  0.09681351,
        0.01135208,  0.00043806,  0.02927307, -0.05133164,  0.00

# **5. SBERT For Analyzing Semantic Textual Similarity (STS)**

In [ ]:
sentences = [
    'The Cat Sits Outside',
    'A man is Playing Guitar',
    'The New Movie is Awesome',
    'The Cat Plays in The Garden',
    'A Woman Watches TV',
    'The New Movie is So Great',
    'Do You Like Pizza?',
    'You Are right. Anyway, Thank You So Much For The Warm Conversation. See You Later!',
    'I Hope You All Had a Great Weekend',
    'I Can See You Have All Got Your School Equipment and Masks, Well Done',
    'I Hate Wearing a Mask, Miss. It is So Uncomfortable!',
    'I Understand, But it is Extremely Important That We Keep Ourselves and Others Safe',
    'Now, Open Your Book and Copy Today Date and Title Off The Board',
    'Miss, I Can Not See The Date Properly',
    'Pupil is A Part of Your Eye That is in Charge of How Much Light Goes Into Our Eye',
    'Better to Finish it Quickly. I Have to Submit The Report to The Manager'
]

embeddings = model.encode(
    sentences,
    convert_to_numpy=True
)
embeddings

array([[ 0.13919115,  0.00302902,  0.04701473, ...,  0.06407333,
        -0.01625155,  0.06362093],
       [ 0.02266129, -0.00138859, -0.00561351, ..., -0.02249004,
         0.08458396, -0.02826073],
       [-0.1004432 , -0.0773927 , -0.00137415, ..., -0.00104971,
         0.07181142,  0.02205477],
       ...,
       [ 0.03780555,  0.01295972,  0.06478752, ...,  0.07640478,
        -0.02517262,  0.01154895],
       [ 0.06587531, -0.03611827, -0.03740051, ...,  0.11304723,
         0.0143741 , -0.01582673],
       [ 0.02521353,  0.06769085, -0.02292574, ..., -0.00371493,
        -0.03869829,  0.05546784]], dtype=float32)

In [ ]:
def cosine_similarity(a,b):
  score = np.dot(a,b) / (norm(a) * norm(b))
  return score

In [ ]:
cosine_similarity(embeddings[0], embeddings[1])

np.float32(0.036330365)

In [ ]:
cosine_similarity(embeddings[3], embeddings[6])

np.float32(0.0899832)

In [ ]:
cosine_similarity(embeddings[5], embeddings[11])

np.float32(0.026413865)

In [ ]:
cosine_scores = util.cos_sim(embeddings, embeddings)
cosine_scores.shape

torch.Size([16, 16])

In [ ]:
pairs = []
for i in range(len(cosine_scores)-1):
  for j in range(i+1, len(cosine_scores)):
    pairs.append(
        {
            'index': [i,j],
            'score': cosine_scores[i][j]
        }
    )


len(pairs)

120

In [ ]:
pca = PCA(n_components=3)
embeddings_reduced = pca.fit_transform(embeddings)
embeddings_reduced

array([[-0.7126087 , -0.13850935,  0.11694185],
       [-0.3860705 , -0.01317526,  0.05827897],
       [ 0.07443842,  0.8253459 , -0.08412052],
       [-0.7984908 , -0.00400809,  0.121751  ],
       [-0.27861634, -0.2981685 , -0.11935136],
       [ 0.07202154,  0.8189547 , -0.09376029],
       [-0.04736597,  0.21521193,  0.27749327],
       [ 0.34761873, -0.19165985, -0.01712299],
       [ 0.29292098,  0.16257063,  0.10488606],
       [ 0.5031339 , -0.24274239,  0.17748699],
       [ 0.2827542 , -0.14595936,  0.45458087],
       [ 0.2508456 , -0.14664206,  0.28647158],
       [ 0.08846514, -0.15671292, -0.66000885],
       [ 0.0778795 , -0.29867452, -0.44929445],
       [ 0.11077157, -0.27864245,  0.3314599 ],
       [ 0.12230284, -0.10718893, -0.50569236]], dtype=float32)

In [ ]:
x, y = embeddings_reduced[:, 0], embeddings_reduced[:, 1]

# KMeans Clustering
n_clusters = min(5, len(sentences))
kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init='auto'
)
labels = kmeans.fit_predict(embeddings_reduced)
cluster_centers = kmeans.cluster_centers_

palette = [
    '#4C72B0', '#DD8452', '#55A868',
    '#C44E52', '#8172B2', '#937860',
    '#DA8BC3', '#8C8C8C', '#CCB974', '#64B5CD'
]

point_colors = [palette[l % len(palette)] for l in labels]

# KNN edges (intra-cluster only)
k = 3
nbrs = NearestNeighbors(
    n_neighbors= k + 1
).fit(embeddings_reduced)

_, indices = nbrs.kneighbors(embeddings_reduced)


edge_traces = []
for i, neighbors in enumerate(indices):
  for j in neighbors[1:]:
    if labels[i] == labels[j]:
      edge_traces.append(go.Scatter(
          x=[x[i],
             x[j]],
          y=[y[i],
             y[j]],
          mode='lines',
          line=dict(color=palette[labels[i] % len(palette)], width=1),
          opacity=0.25,
          hoverinfo='skip',
          showlegend=False
      ))

# KDE per cluster
contour_traces = []
for c in range(n_clusters):
  mask = labels == c
  if mask.sum() < 3:
    continue
  cx, cy = x[mask], y[mask]
  kde = gaussian_kde(np.vstack(
      [cx, cy]
  ), bw_method=0.4)
  xi, yi = np.mgrid[
      x.min(): x.max(): 60j,
      y.min(): y.max(): 60j
  ]

  zi = kde(np.vstack([xi.ravel(), yi.ravel()])).reshape(xi.shape)
  contour_traces.append(go.Contour(
      x=xi[:,0], y=yi[0], z=zi.T,
      colorscale=[
          [0,
           'rgba(0,0,0,0)'],
          [1, palette[c % len(palette)]]
      ],

      opacity=0.18,
      showscale=False,
      ncontours=5,
      contours_coloring='fill',
      hoverinfo='skip',
      showlegend=False
  ))


# cluster cebter labels
center_trace = go.Scatter(
    x=cluster_centers[:,0],
    y=cluster_centers[:, 1],
    mode='markers+text',
    text=[f'Cluster {i}' for i in range(n_clusters)],
    textposition='top center',
    textfont=dict(
        size=11,
        color='white',
        family='Arial Black'
    ),

    marker=dict(
        symbol='diamond',
        size=16,
        color=[palette[i % len(palette)] for i in range(n_clusters)],
        line=dict(color='white', width=2)
    ),

    hoverinfo='text',
    name='Cluster centers',
)


# points
point_trace = go.Scatter(
    x=x,
    y=y,
    mode='markers',
    marker=dict(
        size=11,
        color=point_colors,
        line=dict(color='white', width=1.2),
        opacity=0.9
    ),

    text=[
        f"<b>{s}</b><br>Cluster {labels[i]}"
        for i, s in enumerate(sentences)
    ],

    hovertemplate="%{text}<extra></extra>",
    name='Sentences'
)


# annotation labels (visible on hover via customdata trick)
label_trace = go.Scatter(
    x=x,
    y=y,
    mode='text',
    text=[f'  {s}' for s in sentences],
    textposition='middle right',
    textfont=dict(size=8, color="#333333"),
    hoverinfo='skip',
    showlegend=False,
    visible=True
)


# assemble
fig = go.Figure(
    data=[*contour_traces,
          *edge_traces,
          center_trace,
          point_trace,
          label_trace]
)

fig.update_layout(
    title=dict(
        text="<b>Sentence Embedding Space</b>",
        font=dict(size=17),
        x=0.5,
        xanchor='center'
    ),

    xaxis=dict(
        title='Component 1',
        showgrid=True,
        gridcolor="#ececec",
        zeroline=False,
        showline=True,
        linecolor="#cccccc"
    ),


    yaxis=dict(
        title='Component 2',
        showgrid=True,
        gridcolor='#ececec',
        zeroline=False,
        showline=True,
        linecolor='#cccccc'
    ),

    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        orientation='h',
        y=-0.13,
        bgcolor="rgba(255, 255, 255, 0.7)",
        bordercolor="#dddddd",
        borderwidth=1
    ),

    autosize=True,
    height=1300,
    margin=dict(l=60, r=60, t=80, b=70)
),

# togle labels button
updatemenus = [
    dict(
        type='buttons',
        direction='left',
        x=1.0,
        y=1.08,
        xanchor='right',
        buttons=[
            dict(label='Labels on',
                 method='update',
                 args=[
                     {
                         'visible': [True] * len(contour_traces) + [True] * len(edge_traces) + [True, True, True]
                     }
                 ]),

            dict(
                label='Label Off',
                method='Update',
                args=[
                    {
                        'visible': [True] * len(contour_traces) + [True] * len(edge_traces) + [True, True, False]
                    }
                ]
            )
        ],

        bgcolor='white',
        bordercolor="#cccccc",
        font=dict(size=11)
    )
]

fig.show()

# **6. Preparing The Datasets**

[DOWNLOAD DATASETS](https://www.kaggle.com/datasets/nandini1999/perfume-recommendation-dataset?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkGuidedProjectsIBMGPXX068IEN1371-2022-01-01)

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "final_perfume_data.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "nandini1999/perfume-recommendation-dataset",
  file_path,
  pandas_kwargs={'encoding':'unicode_escape'}
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

# print("First 5 records:", df.head())
df.head(7)

Using Colab cache for faster access to the 'perfume-recommendation-dataset' dataset.


,Name,Brand,Description,Notes,Image URL
0,Tihota Eau de Parfum,Indult,"Rapa Nui for sugar, Tihota is, quite simply, ...","Vanilla bean, musks",https://static.luckyscent.com/images/products/...
1,Sola Parfum,Di Ser,A tribute to the expanse of space extending f...,"Lavender, Yuzu, Lemongrass, Magnolia, Geraniu...",https://static.luckyscent.com/images/products/...
2,Kagiroi Parfum,Di Ser,An aromatic ode to the ancient beauty of Japa...,"Green yuzu, green shikuwasa, sansho seed, cor...",https://static.luckyscent.com/images/products/...
3,Velvet Fantasy Eau de Parfum,Montale,Velvet Fantasy is a solar fragrance where cit...,"tangerine, pink pepper, black coffee, leat...",https://static.luckyscent.com/images/products/...
4,A Blvd. Called Sunset Eau de Parfum,A Lab on Fire,There's no way A Lab On Fire could relocate t...,"Bergamot, almond, violet, jasmine, leather, s...",https://static.luckyscent.com/images/products/...
5,Freckled and Beautiful Eau de Parfum,A Lab on Fire,There's no beauty quite like being young in t...,"Orange flower, neroli, honeysuckle, warm milk...",https://static.luckyscent.com/images/products/...
6,Exit the King Eau de Parfum,Etat Libre d'Orange,"In these tumultuous times, a fragrance that c...","Timur JE, Soap Foam Accord (Aldehydes & Musk)...",https://static.luckyscent.com/images/products/...


In [ ]:
list(df.Notes[0:21])

[' Vanilla bean, musks',
 ' Lavender, Yuzu, Lemongrass, Magnolia, Geranium, Jasmine, Frankincense, Myrrh',
 ' Green yuzu, green shikuwasa, sansho seed, coriander, ylang-ylang, shiso, rosewood, vetiver, hinoki, cypriol, patchouli, agarwood',
 ' tangerine,  pink pepper,  black coffee,  leather,  violet,  jasmine,  lily of the valley,  heliotrope powder,  vanilla,  amber, sandalwood,  toffee,  musk,  oakmoss',
 ' Bergamot, almond, violet, jasmine, leather, sandalwood, vanilla, tonka',
 ' Orange flower, neroli, honeysuckle, warm milk, pastry, salicylates, sandalwood, vanilla bean, heliotrope',
 ' Timur JE, Soap Foam Accord (Aldehydes & Musk), Pink Pepper, Jasmine e-pure, Rose Superessence, Lily-of-the-valley Accord, Patchouli, Moss Absolute, Sandalwood Accord & Orcanox',
 ' Tobacco, hay, elemi, copaiba, olibanum, nutmeg, black pepper, castoreum, atlas cedar, oakmoss, cognac, ambroxan, norlimbanol, cinnamon, cumin seed, ash',
 ' Saffron, champaca, fir balsam, beeswax, amber, damask rose, ro

In [ ]:
df.rename(
    columns={"ï»¿Name": "Name"}, inplace=True
)
df['Name'] = df['Brand'] + " - " + df['Name']
df.drop(
    labels=['Description',
            'Image URL',
            'Brand'], axis=1, inplace=True
)

df.head(8)

,Name,Notes
0,Indult - Tihota Eau de Parfum,"Vanilla bean, musks"
1,Di Ser - Sola Parfum,"Lavender, Yuzu, Lemongrass, Magnolia, Geraniu..."
2,Di Ser - Kagiroi Parfum,"Green yuzu, green shikuwasa, sansho seed, cor..."
3,Montale - Velvet Fantasy Eau de Parfum,"tangerine, pink pepper, black coffee, leat..."
4,A Lab on Fire - A Blvd. Called Sunset Eau de P...,"Bergamot, almond, violet, jasmine, leather, s..."
5,A Lab on Fire - Freckled and Beautiful Eau de ...,"Orange flower, neroli, honeysuckle, warm milk..."
6,Etat Libre d'Orange - Exit the King Eau de Parfum,"Timur JE, Soap Foam Accord (Aldehydes & Musk)..."
7,PRIN - Eshu Extrait,"Tobacco, hay, elemi, copaiba, olibanum, nutme..."


In [ ]:
df.Notes.isnull().sum()

np.int64(80)

In [ ]:
df.dropna(inplace=True)
df.reset_index(
    inplace=True,
    drop=True
)

df.shape

(2111, 2)

In [ ]:
words = [
    'Perfume Oil',
    'Extrait',
    'Travel',
    'Hair',
    'Body',
    'Hand',
    'Intense',
    'Intensivo',
    'Oil'
]

index_to_drop = []
for index, name in enumerate(df.Name):
  if any(word.lower() in name.lower() for word in words):
    index_to_drop.append(index)

In [ ]:
df.drop(index_to_drop, axis=0, inplace=True)
df.reset_index(inplace=True, drop=True)
df.shape

(1612, 2)

# **7. Creating Perfume Notes Embeddings.**

In [ ]:
df.Notes = df.Notes.apply(lambda x: str(x))
notes = df.Notes.to_list()
len(notes)

1612

In [ ]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

note_embeddings = model.encode(
    notes,
    show_progress_bar=True,
    batch_size=64
)

note_embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

array([[-0.0092152 , -0.00588338,  0.07965599, ..., -0.0465106 ,
         0.06299062, -0.04927407],
       [-0.02047073, -0.05532719,  0.08350064, ...,  0.01296031,
         0.07795755,  0.02532452],
       [ 0.02241134, -0.04566668, -0.01234638, ..., -0.04174406,
         0.03946609,  0.11506725],
       ...,
       [-0.03481103, -0.02457182,  0.06051833, ..., -0.07823157,
         0.07495947, -0.01130766],
       [-0.01330567, -0.08048292,  0.09501222, ..., -0.01180693,
         0.06006651, -0.01370788],
       [ 0.02489633, -0.05313787,  0.07445267, ..., -0.04210577,
         0.00254264, -0.01733846]], dtype=float32)

In [ ]:
print(note_embeddings.shape)
print('\n')
print(note_embeddings[0][:1000])

(1612, 384)


[-9.21519939e-03 -5.88337751e-03  7.96559900e-02  1.10818418e-02
  9.20339227e-02 -6.11888319e-02  6.98084012e-02  4.19339053e-02
  1.31228715e-02 -1.08811317e-03  6.51104450e-02 -8.00822377e-02
  1.98498443e-02 -1.38294548e-01 -2.25123726e-02 -7.48573989e-03
  1.24105453e-01  6.44884557e-02  6.84045488e-03 -1.72451567e-02
  4.11716215e-02  1.70634519e-02  8.54872726e-03  6.71903193e-02
 -5.61139658e-02  1.93254333e-02  2.49864291e-02 -2.41799150e-02
 -3.02803852e-02 -1.20585941e-01 -1.73769165e-02  3.18373106e-02
  1.31492205e-02  2.14791838e-02 -1.02081552e-01  3.20526101e-02
 -1.53568741e-02 -2.60394718e-02  6.54244274e-02 -1.51386177e-02
  1.59238949e-02 -6.78843930e-02  1.50188711e-02 -1.73686482e-02
 -6.78988323e-02 -1.90712158e-02  1.83811057e-02 -5.66424839e-02
 -1.30136847e-03 -1.89556982e-02 -1.62308179e-02 -6.89333677e-02
 -3.58953071e-03  2.25383975e-02  1.79827586e-02 -5.44826835e-02
 -1.45096451e-01  9.61968675e-03  4.90830876e-02  6.90806583e-02
 -1.8010279

# **8. Recommending Perfumes Using Cosine Similarity**

In [ ]:
cosine_scores = util.cos_sim(
    note_embeddings,
    note_embeddings
)

cosine_scores.shape

torch.Size([1612, 1612])

In [ ]:
pairs = []

for i in range(len(cosine_scores)-1):
  for j in range(i+1, len(cosine_scores)):
    pairs.append(
        {
            'index': [i,j],
            'score': cosine_scores[i][j]
        }
    )

len(pairs)

1298466

In [ ]:
sorted_pairs = sorted(
    pairs,
    key=lambda x: x['score'],
    reverse=True
)

for pair in sorted_pairs[0:20]:
  i, j = pair['index']
  print(f"{df.iloc[i, 0]} | {df.iloc[j,0]} \n Score: {pair['score']:.3f} \n")

Carthusia - Fiori di Capri Parfum | Carthusia - Fiori di Capri Eau de Parfum 
 Score: 1.000 

Comme des Garcons - 2 Candle | Comme des Garcons - 2 Eau de Parfum 
 Score: 1.000 

Carthusia - Mediterraneo Parfum | Carthusia - Mediterraneo Eau de Parfum 
 Score: 1.000 

Juliette Has a Gun - Not A Perfume Superdose Eau de Parfum | Juliette Has a Gun - Not a Perfume Eau de Parfum 
 Score: 1.000 

Maison Francis Kurkdjian - Gentle fluidity Silver Eau de Parfum | Maison Francis Kurkdjian - gentle Fluidity Gold Eau de Parfum 
 Score: 1.000 

Roja Parfums - Elysium Parfum Cologne | Roja Parfums - Vetiver Parfum Cologne 
 Score: 1.000 

Ormonde Jayne - Ormonde Elixir Parfum | Ormonde Jayne - Ormonde Woman Eau de Parfum 
 Score: 0.984 

PARFUMS DE NICOLAI - Incense Oud Eau de Parfum | PARFUMS DE NICOLAI - Oud Sublime Elixir de Parfum 
 Score: 0.969 

Ormonde Jayne - Ta'if Elixir Parfum | Ormonde Jayne - Ta'if Eau de Parfum 
 Score: 0.960 

J-Scent - Hisui (Jade) Eau de Parfum | J-Scent - Shaft of

# **9. Getting Your Own Perfume Suggestions.**

In [ ]:
my_perfumes = pd.DataFrame(
    [
        [
            'Jo Malone - English Pear & Freesia',
            'Pear, Melon, Freesia, Rose, Musk, Patchouli, Rhuburb, Amber'
        ],

        [
            'Jo Malone - Myrrh & Tonka',
            'Lavender, Myrrh, Tonka Bean, Vanilla, Almond'
        ],

        [
            'Jo Malone - Oud & Bergamot',
            'orange, bergamot, lemon, cedar and oud.'
        ],

        [
            'Guerlain - Néroli Outrenoir',
            'Petitgrain, Bergamot, Tangerine, Lemon, Grapefruit, Tea, Neroli, Orange Blossom, Smoke, Earthy Notes, Vanilla, Benzoin, Ambrette, Oakmoss'
        ],

        [
            'Guerlain - Épices Volées',
            'Coriander, Lemon, Artemisia, Bergamot, Clove, Cardamom, Sage, Bulgarian Rose, Sandalwood, Patchouli, Benzoin, Labdanum.'
        ],

        [
            'Guerlain - Aqua Allegoria Nerolia Vetiver Eau de Toilette',
            'Basil, Vetiver, Fig Accord, Neroli'
        ],

        [
            'Chloe Eau de Parfum',
            'Peony, Litchi, Freesia, Rose, Lily-of-the-valley, Magnolia, Virginia Cedar, Amber.'
        ]
    ],
    columns=df.columns
)


my_perfumes

,Name,Notes
0,Jo Malone - English Pear & Freesia,"Pear, Melon, Freesia, Rose, Musk, Patchouli, R..."
1,Jo Malone - Myrrh & Tonka,"Lavender, Myrrh, Tonka Bean, Vanilla, Almond"
2,Jo Malone - Oud & Bergamot,"orange, bergamot, lemon, cedar and oud."
3,Guerlain - Néroli Outrenoir,"Petitgrain, Bergamot, Tangerine, Lemon, Grapef..."
4,Guerlain - Épices Volées,"Coriander, Lemon, Artemisia, Bergamot, Clove, ..."
5,Guerlain - Aqua Allegoria Nerolia Vetiver Eau ...,"Basil, Vetiver, Fig Accord, Neroli"
6,Chloe Eau de Parfum,"Peony, Litchi, Freesia, Rose, Lily-of-the-vall..."


# **10. Create Perfume Embeddings**

In [ ]:
notes = list(my_perfumes.Notes)

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

my_embeddings = model.encode(
    notes,
    show_progress_bar=True
)
my_embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

array([[ 0.00647851, -0.005287  ,  0.02603362, ..., -0.0057166 ,
         0.07681699,  0.05532723],
       [ 0.00634501, -0.0931145 ,  0.04959133, ..., -0.01040181,
         0.03165305, -0.04884854],
       [-0.04282153,  0.02124743, -0.01599683, ..., -0.03531112,
         0.01644192,  0.02172926],
       ...,
       [-0.01359168, -0.03799743, -0.01397787, ...,  0.01430065,
         0.07208499,  0.08501904],
       [ 0.00803824,  0.00309042,  0.00879411, ..., -0.05991153,
         0.06843214,  0.01933537],
       [-0.01549409,  0.00785855,  0.07611383, ...,  0.01109195,
         0.01923159,  0.02546484]], dtype=float32)

# **11. Produce Cosine Similarity Scores:**

In [ ]:
cosine_scores = util.cos_sim(
    my_embeddings,
    note_embeddings
)

cosine_scores

tensor([[0.4117, 0.5761, 0.6299,  ..., 0.5459, 0.6969, 0.5783],
        [0.5386, 0.7265, 0.5441,  ..., 0.4743, 0.7082, 0.6617],
        [0.2268, 0.5580, 0.5189,  ..., 0.4391, 0.6535, 0.5439],
        ...,
        [0.1975, 0.7170, 0.6387,  ..., 0.4639, 0.6641, 0.6336],
        [0.2132, 0.4485, 0.5167,  ..., 0.3229, 0.3993, 0.3617],
        [0.2359, 0.6094, 0.6097,  ..., 0.5107, 0.6346, 0.5316]])

# **12. Sort The Perfume Similarity Scores**

In [ ]:
my_pairs = []

for i in range(cosine_scores.shape[0]):
  for j in range(cosine_scores.shape[1]):
    my_pairs.append(
        {
            'index': [i,j],
            'score': cosine_scores[i][j]
        }
    )

my_sorted_pairs = sorted(
    my_pairs,
    key=lambda x: x['score'],
    reverse=True
)

my_sorted_pairs

[{'index': [1, 1528], 'score': tensor(0.8580)},
 {'index': [4, 1459], 'score': tensor(0.8565)},
 {'index': [3, 503], 'score': tensor(0.8544)},
 {'index': [3, 530], 'score': tensor(0.8522)},
 {'index': [2, 1345], 'score': tensor(0.8512)},
 {'index': [3, 1075], 'score': tensor(0.8501)},
 {'index': [3, 959], 'score': tensor(0.8482)},
 {'index': [3, 137], 'score': tensor(0.8462)},
 {'index': [3, 692], 'score': tensor(0.8454)},
 {'index': [4, 415], 'score': tensor(0.8450)},
 {'index': [3, 1447], 'score': tensor(0.8407)},
 {'index': [3, 979], 'score': tensor(0.8400)},
 {'index': [4, 1046], 'score': tensor(0.8389)},
 {'index': [3, 445], 'score': tensor(0.8384)},
 {'index': [3, 415], 'score': tensor(0.8382)},
 {'index': [1, 1440], 'score': tensor(0.8353)},
 {'index': [3, 664], 'score': tensor(0.8336)},
 {'index': [4, 1262], 'score': tensor(0.8330)},
 {'index': [1, 1118], 'score': tensor(0.8312)},
 {'index': [3, 574], 'score': tensor(0.8309)},
 {'index': [4, 150], 'score': tensor(0.8307)},
 {'i

In [ ]:
for i in range(
    cosine_scores.shape[0]):

  print(f"Recommended for {my_perfumes.iloc[i,0]}:")
  my_pairs = []

  for j in range(
      cosine_scores.shape[1]):
    my_pairs.append(
        {
            'index': j,
            'score': cosine_scores[i][j]
        }
    )
    my_sorted_pairs = sorted(
        my_pairs,
        key=lambda x: x['score'],
        reverse=True
    )

  for no, pair in enumerate(my_sorted_pairs[:20]):
    print(f" {no+1}. {df.iloc[pair['index'], 0]} (Score: {pair['score']:.3f})")
  print('\n')

Recommended for Jo Malone - English Pear & Freesia:
 1. Alexandre. J - Silver Ombre Eau de Parfum (Score: 0.801)
 2. Montale - Starry Nights Eau de Parfum (Score: 0.788)
 3. BDK Parfums - Bouquet de Hongrie Eau de Parfum (Score: 0.788)
 4. Jovoy Paris - Psychedelique Eau de Parfum (Score: 0.788)
 5. L'Artisan Parfumeur - Champ de Fleurs Eau de Cologne (Score: 0.783)
 6. The Beautiful Mind Series - Precision & Grace Eau de Parfum (Score: 0.778)
 7. Fort & Manle - Amber Absolutely Eau de Parfum (Score: 0.777)
 8. Eight and Bob - Champs de Provence Eau de Parfum (Score: 0.775)
 9. PARFUMS DE NICOLAI - Rose Oud Eau de Parfum (Score: 0.774)
 10. Alexandre. J - Rose Oud Eau de Parfum (Score: 0.769)
 11. Fort & Manle - Impressions de Giverny Eau de Parfum (Score: 0.766)
 12. Xerjoff - V - Accento Eau de Parfum (Score: 0.766)
 13. Pierre Guillaume Paris, Parfumerie Generale - Neroli Ad Astra Eau de Parfum (Score: 0.765)
 14. J-Scent - Yawahada (Soft Skin) Eau de Parfum (Score: 0.765)
 15. Prof

# **What is BERT?**
**BERT (Bidirectional Encoder Representations from Transformers)** adalah model representasi bahasa yang dirancang oleh para peneliti di Google AI language.

**What does BERT do?**
  * berbeda dengan model pre-trained lainnya sebelum BERT, seperti model berbasis fitur ELMo dan model fine-tuning GPT yang melakukan **unidirectional language representation learning,** BERT melatih representasi **bidirectional representations** yang mendalam dari teks tanpa label dengan mengkondisikan secara bersamaan pada konteks kiri dan kanan disemua lapisan.

  * BERT adalah model representasi berbais fine-tuning pertama yang mencapai performa terbaik pada berbagai tugas tingkat kalimat seperti parafrase dan tugas tingkat token seperti **Named Entity Recognition (NER)**, mengungguli banyak arsitektur spesifik tugas.

  * BERT memajukan teknologi terkini untuk 11 tugas NLP. kode dan model pre-trained tersedia: [teks link](https://github.com/google-research/bert.)

# **How Does BERT Work?:**

  * **Masked LM:** sebelum kalimat dimasukkan ke BERT, 15% dari token input dalam setiap kalimat ditutupi secara acak dengan token **[MASK]**. model kemudian memprediksi token yang ditutupi tersebut berdasarkan konteks **left-to-right** atau **right-to-left** yang diberikan oleh token lain yang tidak ditutupi dalam kalimat. hal ini dapat memungkinkan untuk mendapatkan model **bidirectional pretrained.**
  * **Next Sentence Prediction:** selama proses pelatihan BERT, model akan menerima pasangan kalimat sebagai input. saat memilih kalimat A dan B untuk setiap contoh pelatihan 50% kemungkinan B adalah kalimat berikutnya yang sebenarnya dari A dalam dokumen asli dan 50% kemungkinan B adalah kalimat acak dari korpus. model belajar untuk memprediksi apakah B adalah kalimat berikutnya yang sebenarnya dari A. hal ini memungkinkan kita untuk mendapatkan model yang memahami hubungan antara dua kalimat.